# MaskBench: Evaluating Pose Estimation on Masked Video

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/TilburgMultiscaleSummerschool2026/blob/main/thursday-masking/notebooks/03_maskbench_evaluation.ipynb)

**Session:** 13:30 – 15:00 · Thursday Masking Day  
**Question:** Does masking degrade body-language signal quality?  
**Approach:** Compare keypoint predictions on raw vs masked video using RMSE, PCK, velocity, and acceleration

---

MaskBench workflow:
```
Raw video ──► Pose estimator ──► Keypoints_RAW  ┐
                                                 ├─► Compare ──► RMSE / PCK / Velocity
Masked video ► Pose estimator ──► Keypoints_MASKED ┘
```
We treat raw-video keypoints as **ground truth** and masked-video keypoints as **predictions**.

## Part 0 — Setup

In [ ]:
import sys, subprocess

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    subprocess.run(['pip', 'install', '-q',
                    'opencv-python-headless', 'mediapipe>=0.10.9',
                    'numpy', 'pandas', 'matplotlib', 'scipy'], check=True)

import numpy as np
import numpy.ma as ma
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

plt.rcParams.update({
    'figure.facecolor': '#161616', 'axes.facecolor': '#262626',
    'axes.edgecolor':   '#525252', 'axes.labelcolor': '#f4f4f4',
    'xtick.color':      '#c6c6c6', 'ytick.color':     '#c6c6c6',
    'text.color':       '#f4f4f4', 'grid.color':      '#393939',
    'grid.linestyle':   '--',      'font.family':     'sans-serif',
})

print('Setup complete.')

---
## Part 1 — Synthetic ground-truth data

We simulate two keypoint sequences — **raw** (perfect detection) and **masked** (with realistic noise and occasional dropout) — to run through all metrics without needing real video.

In [ ]:
# COCO-17 keypoint names
COCO17_NAMES = [
    'nose', 'L-eye', 'R-eye', 'L-ear', 'R-ear',
    'L-shoulder', 'R-shoulder', 'L-elbow', 'R-elbow',
    'L-wrist', 'R-wrist', 'L-hip', 'R-hip',
    'L-knee', 'R-knee', 'L-ankle', 'R-ankle',
]
N_KP = 17

rng = np.random.default_rng(2026)
T   = 300      # frames (10 s at 30 fps)
FPS = 30.0
W, H = 640, 360  # frame size

# ── Simulate raw keypoints: smooth random walks per keypoint ─────
raw_kps = np.zeros((T, N_KP, 3))   # (T, 17, 3)  x, y, conf

# Rough skeleton layout at frame 0
base_positions = np.array([
    [320, 60],   # nose
    [310, 55],   # L-eye
    [330, 55],   # R-eye
    [295, 65],   # L-ear
    [345, 65],   # R-ear
    [290, 130],  # L-shoulder
    [350, 130],  # R-shoulder
    [270, 190],  # L-elbow
    [370, 190],  # R-elbow
    [255, 250],  # L-wrist
    [385, 250],  # R-wrist
    [300, 230],  # L-hip
    [340, 230],  # R-hip
    [295, 300],  # L-knee
    [345, 300],  # R-knee
    [290, 340],  # L-ankle
    [350, 340],  # R-ankle
], dtype=float)

for kp in range(N_KP):
    walk_x = np.cumsum(rng.normal(0, 1.5, T))
    walk_y = np.cumsum(rng.normal(0, 0.8, T))
    raw_kps[:, kp, 0] = base_positions[kp, 0] + walk_x
    raw_kps[:, kp, 1] = base_positions[kp, 1] + walk_y
    raw_kps[:, kp, 2] = 0.95  # high confidence

# ── Simulate masked keypoints: raw + Gaussian noise + dropouts ──
masked_kps = raw_kps.copy()
noise_std = 8.0   # pixels
masked_kps[:, :, 0] += rng.normal(0, noise_std, (T, N_KP))
masked_kps[:, :, 1] += rng.normal(0, noise_std * 0.7, (T, N_KP))

# Face keypoints (0-4) are harder to detect through masking
masked_kps[:, :5, 0] += rng.normal(0, noise_std * 2, (T, 5))
masked_kps[:, :5, 1] += rng.normal(0, noise_std * 2, (T, 5))

# Random dropout: 15% of frames × face keypoints
dropout = rng.random((T, 5)) < 0.15
masked_kps[dropout, :5, 2] = 0.0

print(f'Synthetic data: {T} frames, {N_KP} keypoints')
print(f'Raw shape:    {raw_kps.shape}')
print(f'Masked shape: {masked_kps.shape}')

---
## Part 2 — Metric implementations

We implement the four core MaskBench metrics from scratch (following the same logic as the production code).

In [ ]:
def euclidean_distance(pred: np.ndarray, gt: np.ndarray) -> np.ndarray:
    """
    Per-frame, per-keypoint Euclidean distance.
    pred, gt: (T, 17, 2) — xy only
    Returns: (T, 17)
    """
    return np.sqrt(((pred - gt) ** 2).sum(axis=-1))


def rmse(pred_kps: np.ndarray, gt_kps: np.ndarray,
         conf_threshold: float = 0.3) -> float:
    """
    Root Mean Square Error across all valid (confident) keypoints.
    pred_kps, gt_kps: (T, 17, 3)  [x, y, conf]
    """
    valid = (gt_kps[:, :, 2] > conf_threshold) & (pred_kps[:, :, 2] > conf_threshold)
    diffs = euclidean_distance(pred_kps[:, :, :2], gt_kps[:, :, :2])
    return float(np.sqrt(np.mean(diffs[valid] ** 2)))


def pck(pred_kps: np.ndarray, gt_kps: np.ndarray,
        threshold_px: float = 20.0,
        conf_threshold: float = 0.3) -> float:
    """
    Percentage of Correct Keypoints: fraction of keypoints within
    `threshold_px` pixels of ground truth.
    """
    valid = (gt_kps[:, :, 2] > conf_threshold) & (pred_kps[:, :, 2] > conf_threshold)
    diffs = euclidean_distance(pred_kps[:, :, :2], gt_kps[:, :, :2])
    correct = (diffs[valid] < threshold_px).sum()
    return float(correct / valid.sum()) if valid.sum() > 0 else 0.0


def velocity_series(kps: np.ndarray, fps: float = 30.0,
                    conf_threshold: float = 0.3) -> np.ndarray:
    """
    Per-keypoint velocity (pixels/s) — shape (T-1, 17).
    NaN where keypoint is missing in either consecutive frame.
    """
    xy   = kps[:, :, :2].astype(float)
    conf = kps[:, :, 2]
    valid = conf > conf_threshold  # (T, 17)

    dx = np.diff(xy, axis=0)            # (T-1, 17, 2)
    speed = np.sqrt((dx ** 2).sum(-1))  # (T-1, 17)  pixels/frame
    speed *= fps                         # pixels/second

    # Mask where either frame has low confidence
    consecutive_valid = valid[:-1] & valid[1:]
    speed[~consecutive_valid] = np.nan
    return speed


def acceleration_series(kps: np.ndarray, fps: float = 30.0,
                         conf_threshold: float = 0.3) -> np.ndarray:
    """
    Per-keypoint acceleration magnitude (pixels/s²) — shape (T-2, 17).
    """
    vel = velocity_series(kps, fps, conf_threshold)  # (T-1, 17)
    acc = np.abs(np.diff(vel, axis=0)) * fps         # (T-2, 17)
    return acc


print('Metrics defined: rmse(), pck(), velocity_series(), acceleration_series()')

---
## Part 3 — Compute metrics on the synthetic data

In [ ]:
RMSE_val = rmse(masked_kps, raw_kps)
PCK_20   = pck(masked_kps, raw_kps, threshold_px=20.0)
PCK_10   = pck(masked_kps, raw_kps, threshold_px=10.0)

vel_raw    = velocity_series(raw_kps,    fps=FPS)
vel_masked = velocity_series(masked_kps, fps=FPS)
acc_raw    = acceleration_series(raw_kps,    fps=FPS)
acc_masked = acceleration_series(masked_kps, fps=FPS)

print('=== MaskBench Results (synthetic) ===')
print(f'RMSE:             {RMSE_val:.2f} px')
print(f'PCK@20px:         {PCK_20:.3f}  ({PCK_20*100:.1f}%)')
print(f'PCK@10px:         {PCK_10:.3f}  ({PCK_10*100:.1f}%)')
print(f'Mean velocity     raw: {np.nanmean(vel_raw):.1f} px/s')
print(f'Mean velocity  masked: {np.nanmean(vel_masked):.1f} px/s')
print(f'Mean accel        raw: {np.nanmean(acc_raw):.1f} px/s²')
print(f'Mean accel     masked: {np.nanmean(acc_masked):.1f} px/s²')

---
## Part 4 — Per-keypoint RMSE breakdown

Not all keypoints are affected equally by masking. Face keypoints (0–4) are typically more degraded because the mask covers the face.

In [ ]:
# Per-keypoint RMSE
valid_mask = (raw_kps[:, :, 2] > 0.3) & (masked_kps[:, :, 2] > 0.3)  # (T, 17)
diffs_sq   = ((masked_kps[:, :, :2] - raw_kps[:, :, :2]) ** 2).sum(-1)  # (T, 17)

per_kp_rmse = np.array([
    np.sqrt(np.mean(diffs_sq[valid_mask[:, k], k])) if valid_mask[:, k].any() else np.nan
    for k in range(N_KP)
])

# ── Plot ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 4))
colors = ['#fa4d56' if i < 5 else '#4589ff' for i in range(N_KP)]
bars = ax.bar(COCO17_NAMES, per_kp_rmse, color=colors, width=0.7)
ax.axhline(RMSE_val, color='#f1c21b', lw=1.5, ls='--', label=f'Overall RMSE={RMSE_val:.1f}px')
ax.set_ylabel('RMSE (pixels)')
ax.set_title('Per-keypoint RMSE: raw vs masked video', pad=10)
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.legend()
ax.grid(True, axis='y', alpha=0.3)

import matplotlib.patches as mpatches
leg_patches = [
    mpatches.Patch(color='#fa4d56', label='Face keypoints (masked)'),
    mpatches.Patch(color='#4589ff', label='Body keypoints'),
]
ax.legend(handles=leg_patches + ax.get_legend_handles_labels()[0][1:],
           labels=['Face keypoints (masked)', 'Body keypoints',
                   f'Overall RMSE={RMSE_val:.1f}px'],
           fontsize=9)
plt.tight_layout()
plt.show()

---
## Part 5 — Velocity distribution comparison

Do velocity distributions look the same on raw vs masked video? If masking degrades pose accuracy, we'd expect higher variance in the masked velocity estimates.

In [ ]:
# Focus on wrist keypoints (indices 9, 10) — most motion-relevant
WRIST_KPS = [9, 10]
wrist_names = [COCO17_NAMES[i] for i in WRIST_KPS]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, kp_idx, name in zip(axes, WRIST_KPS, wrist_names):
    v_raw    = vel_raw[:, kp_idx]
    v_masked = vel_masked[:, kp_idx]

    # Drop NaNs
    v_raw    = v_raw[~np.isnan(v_raw)]
    v_masked = v_masked[~np.isnan(v_masked)]

    bins = np.linspace(0, max(v_raw.max(), v_masked.max()) * 1.05, 40)
    ax.hist(v_raw,    bins=bins, alpha=0.6, color='#42be65', label='Raw video')
    ax.hist(v_masked, bins=bins, alpha=0.6, color='#ff7eb6', label='Masked video')
    ax.axvline(v_raw.mean(),    color='#42be65', lw=2, ls='--')
    ax.axvline(v_masked.mean(), color='#ff7eb6', lw=2, ls='--')
    ax.set_title(f'{name} velocity distribution', fontsize=10)
    ax.set_xlabel('Velocity (px/s)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Velocity distributions: raw vs masked video (wrists)',
             fontsize=11, color='#f4f4f4')
plt.tight_layout()
plt.show()

---
## Part 6 — Kinematic time series: wrist trajectory

Plot raw vs masked wrist position over time to visually inspect how well the masked signal tracks the original.

In [ ]:
t = np.arange(T) / FPS  # seconds

fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=True)

kp_configs = [
    (9,  'L-wrist', 'x', 0),
    (9,  'L-wrist', 'y', 1),
    (10, 'R-wrist', 'x', 0),
    (10, 'R-wrist', 'y', 1),
]

for ax, (kp_idx, kp_name, axis, xy) in zip(axes.flat, kp_configs):
    raw_vals    = raw_kps[:, kp_idx, xy]
    masked_vals = masked_kps[:, kp_idx, xy]
    # Mask low-confidence frames
    raw_vals    = np.where(raw_kps[:, kp_idx, 2] > 0.3, raw_vals, np.nan)
    masked_vals = np.where(masked_kps[:, kp_idx, 2] > 0.3, masked_vals, np.nan)

    ax.plot(t, raw_vals,    color='#42be65', lw=1.5, alpha=0.9, label='Raw')
    ax.plot(t, masked_vals, color='#ff7eb6', lw=1.0, alpha=0.7, ls='--', label='Masked')
    ax.set_title(f'{kp_name} – {axis}', fontsize=9)
    ax.set_ylabel('Position (px)')
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3)

axes[1, 0].set_xlabel('Time (s)')
axes[1, 1].set_xlabel('Time (s)')
plt.suptitle('Keypoint trajectories: raw vs masked', fontsize=12, color='#f4f4f4')
plt.tight_layout()
plt.show()

---
## Part 7 — Multi-model comparison table

MaskBench compares multiple masking strategies side-by-side. Here we simulate results for three configurations and build a summary table — exactly what the MaskBench report generates.

In [ ]:
# Simulate three masking conditions
def simulate_masked(noise_scale: float, dropout_rate: float,
                    face_penalty: float = 2.0) -> np.ndarray:
    kps = raw_kps.copy()
    kps[:, :, 0] += rng.normal(0, noise_scale, (T, N_KP))
    kps[:, :, 1] += rng.normal(0, noise_scale * 0.7, (T, N_KP))
    kps[:, :5, 0] += rng.normal(0, noise_scale * face_penalty, (T, 5))
    kps[:, :5, 1] += rng.normal(0, noise_scale * face_penalty, (T, 5))
    drop = rng.random((T, 5)) < dropout_rate
    kps[drop, :5, 2] = 0.0
    return kps


conditions = {
    'Blur (σ=30)':      simulate_masked(noise_scale=8,  dropout_rate=0.15),
    'Pixelate (15px)':  simulate_masked(noise_scale=12, dropout_rate=0.20),
    'Solid fill':       simulate_masked(noise_scale=5,  dropout_rate=0.30, face_penalty=4.0),
}

rows = []
for name, kps in conditions.items():
    rows.append({
        'Condition':     name,
        'RMSE (px)':     round(rmse(kps, raw_kps), 2),
        'PCK@20px (%)':  round(pck(kps, raw_kps, 20.0) * 100, 1),
        'PCK@10px (%)':  round(pck(kps, raw_kps, 10.0) * 100, 1),
        'Mean vel (px/s)': round(np.nanmean(velocity_series(kps, FPS)), 1),
        'Mean acc (px/s²)': round(np.nanmean(acceleration_series(kps, FPS)), 1),
    })

df = pd.DataFrame(rows).set_index('Condition')
display(df)

In [ ]:
# ── Visualise as heatmap ──────────────────────────────────────────
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

# Normalize each metric column 0→1 (0=best, 1=worst)
df_norm = df.copy()
for col in df.columns:
    col_range = df[col].max() - df[col].min()
    if col_range > 0:
        # For PCK, higher=better, so invert
        if 'PCK' in col:
            df_norm[col] = 1 - (df[col] - df[col].min()) / col_range
        else:
            df_norm[col] = (df[col] - df[col].min()) / col_range

fig, ax = plt.subplots(figsize=(10, 3))
im = ax.imshow(df_norm.values.astype(float), cmap='RdYlGn_r',
               vmin=0, vmax=1, aspect='auto')

ax.set_xticks(range(len(df.columns)))
ax.set_xticklabels(df.columns, rotation=35, ha='right', fontsize=9)
ax.set_yticks(range(len(df.index)))
ax.set_yticklabels(df.index, fontsize=9)

# Annotate with raw values
for i in range(len(df.index)):
    for j in range(len(df.columns)):
        ax.text(j, i, str(df.values[i, j]),
                ha='center', va='center', fontsize=9,
                color='#161616' if 0.3 < df_norm.values[i, j] < 0.7 else '#f4f4f4')

plt.colorbar(im, ax=ax, label='Relative degradation (green=best)')
ax.set_title('MaskBench: multi-condition comparison heatmap', pad=10)
plt.tight_layout()
plt.show()

---
## Part 8 — Running on real video

To apply this to your own data you need:
1. Raw video + paired masked video (same content, same duration)
2. Run a pose estimator (MediaPipe or YOLO) on both
3. Export COCO-17 JSON from Masked-Piper v2 (`--export-coco`)
4. Feed both JSON files into the metrics above

The cells below show how to load COCO-17 JSON produced by Masked-Piper v2.

In [ ]:
import json


def load_coco17_json(json_path: Path) -> np.ndarray:
    """
    Load a COCO-17 pose JSON produced by Masked-Piper v2.
    Returns array of shape (T, 17, 3) — x, y, confidence.
    Assumes single-person video; takes the highest-confidence person per frame.
    """
    with open(json_path) as f:
        data = json.load(f)

    # COCO JSON structure: list of annotations per frame
    # Each annotation: {frame_id, keypoints: [x, y, v, x, y, v, ...] (51 values)}
    frames_by_id = {}
    for ann in data.get('annotations', []):
        fid = ann['frame_id']
        kps_flat = ann['keypoints']   # 51 values
        kps = np.array(kps_flat).reshape(17, 3)
        # Keep highest-score person per frame
        score = kps[:, 2].mean()
        if fid not in frames_by_id or score > frames_by_id[fid]['score']:
            frames_by_id[fid] = {'kps': kps, 'score': score}

    n_frames = max(frames_by_id.keys()) + 1
    out = np.zeros((n_frames, 17, 3))
    for fid, val in frames_by_id.items():
        out[fid] = val['kps']
    return out


print('load_coco17_json() defined.')
print()
print('Example usage:')
print("  raw_kps    = load_coco17_json(Path('output/clip__pose.json'))")
print("  masked_kps = load_coco17_json(Path('output/clip__blur_masked__pose.json'))")
print("  print(f'RMSE: {rmse(masked_kps, raw_kps):.2f} px')")

---
## Exercise

1. In `simulate_masked()`, increase `noise_scale` to 25 — how does RMSE change?
2. Change `threshold_px` in `pck()` from 20 to 5 — what does this tell you about strict vs lenient evaluation?
3. **Discussion**: MaskBench compares pose from raw vs masked video. Why does a **high RMSE on face keypoints** not necessarily mean the study data is compromised?

---
## Key takeaways

| Metric | What it tells you |
|---|---|
| **RMSE** | Average pixel error across all keypoints — lower is better |
| **PCK** | % of frames where keypoint is within N pixels — intuitive |
| **Velocity** | Motion magnitude — should be similar to raw if masking is successful |
| **Acceleration** | Signal dynamics — sensitive to jitter introduced by masking noise |

MaskBench source: [github.com/synapsis-lab/maskbench](https://github.com/synapsis-lab/maskbench)